# 07 - model testing (sample)

final inference pipeline: load model, run on held-out test sample, export results.

In [1]:
import joblib, pandas as pd, numpy as np, librosa
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import sys
sys.path.append('../../')
from src.config.settings import SAMPLES, FIGURES_DIR
from src.utils.helpers import extract_mfcc

In [2]:
df = pd.read_csv(SAMPLES / "sample_labels.csv")
svm = joblib.load(SAMPLES / "svm_baseline_sample.pkl")

In [3]:
test = df[df['split']=='test']
print(f"testing on {len(test)} files (actors {sorted(test['actor'].unique())})")

testing on 104 files (actors [np.int64(23)])


In [4]:
X_test = np.array([extract_mfcc(fp) for fp in test['filepath']])
y_test = test['emotion_code'].values


In [5]:
scaler = StandardScaler().fit(X_test)
y_pred = svm.predict(scaler.transform(X_test))

In [6]:
acc = accuracy_score(y_test, y_pred)
print(f"test accuracy: {acc*100:.1f}%")

test accuracy: 30.8%


In [7]:
results = test.copy()
results['predicted'] = y_pred
results['correct'] = (y_test == y_pred).astype(int)

In [8]:
csv_path = SAMPLES / 'predictions_sample.csv'
results.to_csv(csv_path, index=False)
print(f"predictions saved: {csv_path}")

predictions saved: /mnt/d/career/projects/lightweight-speech-emotion-recognition-on-open-datasets/data/samples/predictions_sample.csv


per-speaker accuracy:

In [9]:
for actor in sorted(test['actor'].unique()):
    sub = results[results['actor']==actor]
    a = sub['correct'].mean() * 100
    print(f"  Actor_{actor:02d}: {a:.1f}% ({sub['correct'].sum()}/{len(sub)})")

  Actor_23: 30.8% (32/104)


testing complete. predictions exported for midterm report analysis.